# Production postprocessing check
CPU only. Run the final inference postprocessor on both real pilot outputs and require the exact prior CSV hash.
No new image inference, labels, training, or submission.


In [ ]:
import base64,hashlib,io,os,subprocess,sys,zipfile
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']=''
os.environ['POLARS_PREFER_PKG']='32'
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='0b797c8d80f42d0a0326b0a532083f830131fb77367b039e1a05b34840cb078a'
package=Path('/kaggle/working/visual_submission_smoke_package');package.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        if not (package/name).resolve().is_relative_to(package.resolve()):raise ValueError(name)
    archive.extractall(package)
support=[Path('/kaggle/input/biohub-tracking-support-pack-50ep-v1'),Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')]
wheels=sorted({str(p.parent) for root in support if root.is_dir() for p in root.rglob('*.whl')})
if not wheels:raise RuntimeError('Offline dependency wheels missing')
cmd=[sys.executable,'-m','pip','install','--no-index']
for folder in wheels:cmd+=['--find-links',folder]
subprocess.run(cmd+['tracksdata','polars>=1.36','zarr>=3.0.10,<4','geff-spec<1.2'],check=True)
env=dict(os.environ,PYTHONPATH=os.pathsep.join([str(package/'src'),str(package/'scripts')]),OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='1')
subprocess.run([sys.executable,'-u',str(package/'scripts/visual_submission_smoke_runner.py'),str(package)],env=env,check=True)
